In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label_num'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

def predict_message(message):
    message_tfidf = vectorizer.transform([message])
    prediction = model.predict(message_tfidf)[0]
    probability = model.predict_proba(message_tfidf)[0]
    label = "SPAM" if prediction == 1 else "HAM"
    confidence = max(probability) * 100
    print(f"{label} ({confidence:.1f}% confident) → {message}")

predict_message("Free entry in 2 a wkly comp to win FA Cup final tkts!")
predict_message("WINNER!! You have been selected to receive a £900 prize reward!")
predict_message("Txt to claim your prize 83600")
predict_message("Hey are we still meeting tomorrow?")

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.81      0.90       149

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115

HAM (53.2% confident) → Free entry in 2 a wkly comp to win FA Cup final tkts!
SPAM (51.1% confident) → WINNER!! You have been selected to receive a £900 prize reward!
SPAM (89.1% confident) → Txt to claim your prize 83600
HAM (96.8% confident) → Hey are we still meeting tomorrow?


In [2]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("Saved!")

Saved!


In [3]:
import os
print(os.getcwd())

d:\spam-mlpjct
